<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-PT/blob/main/lingustic_structure_and_interpretability/dependency_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: The Collapse of Dependencies

**Learning Objective:** Attack the fragility of syntactic parsing using real-world linguistic ambiguity.

Statistical Part-of-Speech (POS) Tagging models assign tags based on the probabilistic context of surrounding words. They do not "understand" the sentence. When faced with structurally ambiguous language—specifically "Garden-path sentences" where a word's typical part of speech is subverted—the model's statistical proximity heuristic fails against the structural grammar.

Today, we will intentionally trick the parser to expose this blind spot.

In [3]:
# Phase 1: Environment Setup
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import spacy
from spacy import displacy

# Load the small English pipeline
nlp = spacy.load("en_core_web_sm")
print("Environment and NLP model initialized.")

def audit_syntax(text):
    doc = nlp(text)
    print(f"Parsing: '{text}'")
    print("-" * 50)
    for token in doc:
        # Print the Token, its Part-of-Speech, and its Dependency Tag
        print(f"Token: {token.text: <12} | POS: {token.pos_: <6} | Dependency: {token.dep_}")

    print("-" * 50)
    # Render the dependency graph for visual auditing
    html = displacy.render(doc, style="dep", jupyter=False, options={'distance': 90})
    display(HTML(html))

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Environment and NLP model initialized.


### Phase 2: The Baseline Test

First, we will run a standard, highly predictable sentence through our parsing function. The statistical model will easily classify the nouns, adjectives, and verbs because they appear in their most mathematically common configuration

In [4]:
from IPython.display import HTML
# The model succeeds
standard_sentence = "The old man is walking."
audit_syntax(standard_sentence)

Parsing: 'The old man is walking.'
--------------------------------------------------
Token: The          | POS: DET    | Dependency: det
Token: old          | POS: ADJ    | Dependency: amod
Token: man          | POS: NOUN   | Dependency: nsubj
Token: is           | POS: AUX    | Dependency: aux
Token: walking      | POS: VERB   | Dependency: ROOT
Token: .            | POS: PUNCT  | Dependency: punct
--------------------------------------------------


### Phase 3: The Adversarial Attack

Now we introduce a classic Garden-path sentence: *"The old man the boat."* To a human taking a second glance, "man" is the verb (meaning to operate or staff) and "old" is a noun referring to elderly people. Look at the topological graph and the POS tags generated below to see what the machine thinks.

In [5]:
# The model fails
garden_path_sentence = "The old man the boat."
audit_syntax(garden_path_sentence)

Parsing: 'The old man the boat.'
--------------------------------------------------
Token: The          | POS: DET    | Dependency: det
Token: old          | POS: ADJ    | Dependency: amod
Token: man          | POS: NOUN   | Dependency: ROOT
Token: the          | POS: DET    | Dependency: det
Token: boat         | POS: NOUN   | Dependency: appos
Token: .            | POS: PUNCT  | Dependency: punct
--------------------------------------------------


---
### 🛑 YOUR FORUM ASSIGNMENT

Did you look at the graph? The machine relies on statistical proximity. Because the word "man" acts as a noun in 99% of its training data, the algorithm stubbornly assigns it the `NOUN` tag, leaving the sentence without a structural verb and completely destroying the dependency tree.

**Your Task:**
1. Analyze the logical failure on the token "man" using the data outputted in Phase 3.
2. Create **three** deceptive garden-path sentences of your own in English. They must be grammatically correct sentences that trick the `en_core_web_sm` model into assigning the wrong POS tag to the core semantic action.
3. Fix the parsing outputs of your three sentences by introducing a single punctuation mark (like a comma) or slightly altering the surrounding context to force the probabilistic model to correct the dependency tree.

**To submit in the forum:**
Post the `displacy` POS tag evidence for the word "man" in Phase 3. Then, present your three original garden-path sentences. Show the output of the model *failing* on your sentences, and then show the output of the model *succeeding* after your punctuation fix.

Explain fundamentally why statistical token-by-token processing will always struggle with global sentence structure unless augmented by attention mechanisms (which we will cover next week).